In [1]:
import pandas as pd
import tqdm
import requests
import json

import utils

2025-02-24 13:53:14.888878: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
df = pd.read_csv('Data_FailureFixExplanation.csv')

# API provided by running llm using Ollama locally
url = "http://localhost:11434/api/generate"
headers = {
    "Content-Type": "application/json"
}
model_deepseek_llm = "deepseek-llm:7b"
model_llama = "llama3.2:3b"
data = {
    "model": model_deepseek_llm,
    "prompt": "",
    "stream": False
}

In [3]:
prompt_task_professional = "###Task:\nYou are a professional software developer. You are given a number of explanations describing the root cause of a software failure. Based on the given explanations, write a single explanation that contains all the information required to understand teh root cause of the bug. The explanation should be succinct and without redundant information"

prompt_input = "### Input:\n\nHere are the failure explanations:\n\n"

prompt_output = "### Output:\nFormat your response in valid JSON format with a single field 'explanation' of type string containing your generated explanation."

In [4]:
def generate_explanations(explanations, url, headers, data, prompt_task_professional, prompt_input, prompt_output=''):
    explanation_text = explanations['Answer.explanation'].to_list()
    input = ''
    responses = []

    for expl in tqdm.tqdm(explanation_text):
        input = input + "'''\n" + expl + "\n'''\n\n"
        prompt = prompt_task_professional + prompt_input + input + prompt_output
        data['prompt'] = prompt
        response = requests.post(url, headers=headers, data=json.dumps(data))
        if (response.status_code == 200):
            responses.append({'prompt': prompt, 'result': json.loads(response.text)['response']})
    return responses

In [ ]:
data_llama = data.copy()
data_llama['model'] = model_llama
responses = generate_explanations(explanations_sorted_df, url, headers, data, prompt_task_professional, prompt_input)
responses_llama = generate_explanations(explanations_sorted_df, url, headers, data_llama, prompt_task_professional, prompt_input)

In [ ]:
schema = {
    "$schema": "ase-schema",
    "title": "Explanation",
    "description": "A failure explanation",
    "type": "object",
    "properties": {
        "explanation": {
            "description": "The generated explanation",
            "type": "string"
        }
    },
    "required": ["explanation"]
}
# data_json = {
#     "model": model_deepseek_llm,
#     "prompt": "",
#     "stream": False,
#     "format": "json"
# }
data_json = {
    "model": model_deepseek_llm,
    "prompt": "",
    "stream": False,
    "format": schema
}
responses_json = generate_explanations(explanations_sorted_df, url, headers, data_json, prompt_task, prompt_input, prompt_output)